# PIDA-RL Diabetes — 00. Setup reproducible en Google Colab

Este es el **único notebook de preparación del entorno**. Su propósito es dejar una sesión de Google Colab lista para ejecutar el proyecto de forma reproducible, sin entrenar modelos, modificar datos, ni realizar commits automáticos.

## Principios de esta nueva etapa

- Google Drive es el almacenamiento persistente de datos, modelos, resultados y figuras.
- `/content/pida-rl-datos-publicos` es una copia temporal del repositorio durante la sesión de Colab.
- Las rutas se resuelven de forma robusta para evitar depender de si Drive quedó montado en `/content/drive` o `/content/gdrive`.
- Este notebook **no falla** si aún no existe una cohorte o un resultado: esos artefactos se crearán en los notebooks posteriores.
- Nunca escribas, imprimas o guardes tokens, contraseñas o credenciales en celdas, salidas o commits.


## 1. Parámetros del proyecto

Modifica únicamente `REPO_URL` si cambia la ubicación del repositorio. Las rutas derivadas se usarán en todos los notebooks posteriores.

In [1]:
from pathlib import Path
import os
import sys
import subprocess
import importlib.util

PROJECT_NAME = 'pida-rl-datos-publicos'
REPO_URL = 'https://github.com/JssSalas/pida-rl-datos-publicos.git'
PROJECT_ROOT = Path('/content') / PROJECT_NAME

# Déjalo como None para respetar la rama actual del repositorio.
WORK_BRANCH = None

print('Proyecto local:', PROJECT_ROOT)
print('Repositorio remoto:', REPO_URL)
print('Rama solicitada:', WORK_BRANCH or 'rama actual')


Proyecto local: /content/pida-rl-datos-publicos
Repositorio remoto: https://github.com/JssSalas/pida-rl-datos-publicos.git
Rama solicitada: rama actual


## 2. Montar Google Drive sin conflictos

Esta celda reutiliza Drive si ya está montado. Si `/content/drive` contiene archivos locales y no está montado, usa `/content/gdrive` como punto alternativo.

In [2]:
DRIVE_MOUNT_CANDIDATES = [Path('/content/drive'), Path('/content/gdrive')]
DRIVE_ROOT = next(
    (path for path in DRIVE_MOUNT_CANDIDATES if (path / 'MyDrive').is_dir()),
    None
)

if DRIVE_ROOT is None:
    from google.colab import drive

    preferred = Path('/content/drive')
    alternative = Path('/content/gdrive')

    if preferred.exists() and any(preferred.iterdir()):
        mount_point = alternative
    else:
        mount_point = preferred

    if mount_point.exists() and any(mount_point.iterdir()):
        raise RuntimeError(
            f'El punto de montaje {mount_point} contiene archivos y no es Google Drive. '
            'Reinicia la sesión de Colab o libera esa carpeta antes de montar Drive.'
        )

    drive.mount(str(mount_point), force_remount=False)
    DRIVE_ROOT = mount_point

MYDRIVE_ROOT = DRIVE_ROOT / 'MyDrive'
PERSISTENT_ROOT = MYDRIVE_ROOT / 'PIDA-RL-Diabetes'

print('Google Drive disponible en:', DRIVE_ROOT)
print('Raíz persistente del proyecto:', PERSISTENT_ROOT)


Mounted at /content/drive
Google Drive disponible en: /content/drive
Raíz persistente del proyecto: /content/drive/MyDrive/PIDA-RL-Diabetes


## 3. Crear estructura persistente

Se crean solamente carpetas. No se crean datos sintéticos, cohortes vacías ni modelos ficticios.

In [3]:
DATA_ROOT = PERSISTENT_ROOT / 'data'
RAW_DATA_DIR = DATA_ROOT / 'raw'
INTERIM_DATA_DIR = DATA_ROOT / 'interim'
PROCESSED_DATA_DIR = DATA_ROOT / 'processed'
MODELS_DIR = PERSISTENT_ROOT / 'models'
RESULTS_DIR = PERSISTENT_ROOT / 'results'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR = RESULTS_DIR / 'figures'
LOGS_DIR = PERSISTENT_ROOT / 'logs'
EXPORTS_DIR = PERSISTENT_ROOT / 'exports'

persistent_dirs = [
    RAW_DATA_DIR, INTERIM_DATA_DIR, PROCESSED_DATA_DIR,
    MODELS_DIR, TABLES_DIR, FIGURES_DIR, LOGS_DIR, EXPORTS_DIR
]

for directory in persistent_dirs:
    directory.mkdir(parents=True, exist_ok=True)

print('Carpetas persistentes listas:')
for directory in persistent_dirs:
    print(' -', directory)


Carpetas persistentes listas:
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/raw
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/interim
 - /content/drive/MyDrive/PIDA-RL-Diabetes/data/processed
 - /content/drive/MyDrive/PIDA-RL-Diabetes/models
 - /content/drive/MyDrive/PIDA-RL-Diabetes/results/tables
 - /content/drive/MyDrive/PIDA-RL-Diabetes/results/figures
 - /content/drive/MyDrive/PIDA-RL-Diabetes/logs
 - /content/drive/MyDrive/PIDA-RL-Diabetes/exports


## 4. Instalar dependencias

Las dependencias se instalan en la sesión actual. Si Colab indica `Requirement already satisfied`, no es un error.

In [4]:
PACKAGES = [
    'numpy',
    'pandas',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'pyreadstat',
    'gymnasium',
    'stable-baselines3',
    'streamlit',
    'plotly',
    'pyyaml',
    'joblib'
]

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', *PACKAGES
])

print('Dependencias disponibles.')


Dependencias disponibles.


## 5. Clonar o reutilizar el repositorio

El repositorio vive temporalmente en `/content`. Esta celda no hace `pull`, `push`, `commit` ni cambia de rama automáticamente.

In [5]:
if not PROJECT_ROOT.exists():
    subprocess.check_call(['git', 'clone', REPO_URL, str(PROJECT_ROOT)])
    print('Repositorio clonado.')
elif not (PROJECT_ROOT / '.git').is_dir():
    raise RuntimeError(
        f'La ruta existe pero no es un repositorio Git: {PROJECT_ROOT}. '
        'Renombra o elimina esa carpeta local antes de continuar.'
    )
else:
    print('Repositorio local encontrado; no se vuelve a clonar.')

os.chdir(PROJECT_ROOT)
print('Directorio actual:', Path.cwd())


Repositorio clonado.
Directorio actual: /content/pida-rl-datos-publicos


## 6. Estado de Git y código fuente

La identidad de Git es opcional para ejecutar el proyecto. Nunca configures tokens mediante `git config`.

In [6]:
def run_git(*args, check=True):
    result = subprocess.run(
        ['git', *args],
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
        check=check
    )
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    return result

run_git('status', '--short')
current_branch = run_git('branch', '--show-current').stdout.strip()
print('Rama activa:', current_branch or '(sin rama detectada)')
run_git('log', '--oneline', '-3', check=False)

if WORK_BRANCH is not None and current_branch != WORK_BRANCH:
    raise RuntimeError(
        f'La rama activa es {current_branch!r}; se esperaba {WORK_BRANCH!r}. '
        'Cámbiala manualmente antes de ejecutar el resto del flujo.'
    )

SRC_ROOT = PROJECT_ROOT / 'src'
SRC_ROOT.mkdir(parents=True, exist_ok=True)

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print('SRC_ROOT agregado a sys.path:', SRC_ROOT)


main
Rama activa: main
ba2bbba fix: corrije la codificación y se imputan las variables binarias; recalcula las categorías de riesgo
2f9cdaa fix: corrige código especial 99 en a0302 y a0303num; recalcula anios_con_diabetes
d1e434a feat: calibra categoría de riesgo con percentiles; documenta distribución final de cohorte
SRC_ROOT agregado a sys.path: /content/pida-rl-datos-publicos/src


## 7. Validar rutas de entrada y salida

El setup solo reporta el estado de los artefactos. La cohorte se crea en el notebook 03; los modelos se entrenan después de validar el MDP.

In [7]:
RAW_PATTERNS = ('*.sav', '*.csv', '*.xlsx', '*.xls', '*.parquet')
raw_files = []
for pattern in RAW_PATTERNS:
    raw_files.extend(RAW_DATA_DIR.rglob(pattern))

COHORT_PATH = PROCESSED_DATA_DIR / 'cohorte_diabetes.csv'

print('Archivos fuente detectados:', len(raw_files))
for path in sorted(raw_files)[:20]:
    print(' -', path.name)
if len(raw_files) > 20:
    print(' - ...')

print('Cohorte procesada:', COHORT_PATH)
print('Existe cohorte:', COHORT_PATH.is_file())
print('Directorio de modelos:', MODELS_DIR)
print('Directorio de tablas:', TABLES_DIR)
print('Directorio de figuras:', FIGURES_DIR)


Archivos fuente detectados: 0
Cohorte procesada: /content/drive/MyDrive/PIDA-RL-Diabetes/data/processed/cohorte_diabetes.csv
Existe cohorte: False
Directorio de modelos: /content/drive/MyDrive/PIDA-RL-Diabetes/models
Directorio de tablas: /content/drive/MyDrive/PIDA-RL-Diabetes/results/tables
Directorio de figuras: /content/drive/MyDrive/PIDA-RL-Diabetes/results/figures


## 8. Diagnóstico de versiones

Registra estas versiones en la bitácora del proyecto cuando ejecutes experimentos reproducibles.

In [8]:
import numpy as np
import pandas as pd
import gymnasium
import stable_baselines3
import pyreadstat
import sklearn

versions = {
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit-learn': sklearn.__version__,
    'gymnasium': gymnasium.__version__,
    'stable-baselines3': stable_baselines3.__version__,
    'pyreadstat': pyreadstat.__version__
}

for package, version in versions.items():
    print(f'{package}: {version}')


python: 3.13.15
numpy: 2.1.3
pandas: 2.2.3
scikit-learn: 1.6.1
gymnasium: 1.3.0
stable-baselines3: 2.9.0
pyreadstat: 1.3.6


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 9. Flujo de ejecución oficial

El orden evita evaluar modelos antes de entrenarlos y evita ejecutar análisis de equidad antes de contar con una evaluación conjunta.

1. `01_fuentes_y_carga.ipynb` — carga y trazabilidad de fuentes públicas.
2. `02_entendimiento_datos.ipynb` — calidad, estructura y exploración de datos.
3. `03_preparacion_cohorte.ipynb` — cohorte final y validación de valores faltantes/no finitos.
4. `04_definicion_mdp.ipynb` — entorno Gymnasium, estado, acciones, recompensas y `check_env`.
5. `05_entrenamiento_dqn.ipynb` — entrenamiento y persistencia de DQN.
6. `06_entrenamiento_q_learning.ipynb` — entrenamiento y persistencia de Q-learning tabular.
7. `07_entrenamiento_ppo.ipynb` — validación y entrenamiento de PPO.
8. `08_evaluacion_modelos.ipynb` — evaluación homogénea: aleatoria, reglas, DQN, Q-learning y PPO.
9. `09_equidad_y_subgrupos.ipynb` — cobertura y brechas por riesgo, sexo y edad.
10. `10_resultados_finales.ipynb` — tablas, visualizaciones y conclusiones técnicas.
11. `11_dashboard_streamlit.ipynb` — generación, prueba y despliegue local de la app Streamlit.
12. `12_documentacion_reproducibilidad.ipynb` — README, CRISP-DM, registro de experimentos y checklist de entrega.

### Convención de artefactos persistentes

- Cohorte: `data/processed/cohorte_diabetes.csv`
- Modelos: `models/`
- Tablas de evaluación: `results/tables/`
- Figuras: `results/figures/`
- Bitácoras y TensorBoard: `logs/`

Los notebooks posteriores deben obtener las rutas desde esta misma convención y verificar la existencia de cada insumo antes de usarlo.

## 10. Checklist final del setup

Ejecuta esta celda antes de continuar. Si la cohorte aún no existe, es normal: se creará en el notebook 03.

In [9]:
checks = {
    'Drive montado': MYDRIVE_ROOT.is_dir(),
    'Raíz persistente': PERSISTENT_ROOT.is_dir(),
    'Repositorio Git': (PROJECT_ROOT / '.git').is_dir(),
    'Código fuente src': SRC_ROOT.is_dir(),
    'Datos raw': RAW_DATA_DIR.is_dir(),
    'Datos processed': PROCESSED_DATA_DIR.is_dir(),
    'Modelos': MODELS_DIR.is_dir(),
    'Tablas': TABLES_DIR.is_dir(),
    'Figuras': FIGURES_DIR.is_dir(),
}

for name, ok in checks.items():
    print(('✓' if ok else '✗'), name)

assert checks['Drive montado']
assert checks['Repositorio Git']
assert checks['Código fuente src']

print('\nSetup terminado. Continúa con 01_fuentes_y_carga.ipynb.')


✓ Drive montado
✓ Raíz persistente
✓ Repositorio Git
✓ Código fuente src
✓ Datos raw
✓ Datos processed
✓ Modelos
✓ Tablas
✓ Figuras

Setup terminado. Continúa con 01_fuentes_y_carga.ipynb.
